# 第19章 绘图及交互（跟随代码）

> 《OpenCV轻松入门：面向Python》（李立宗）第 19 章的跟随代码 notebook。
> 全量收录原书代码：例 19.1～19.14 共 14 例及章内独立演示（isClosed=False 折线、鼠标事件清单）；缺失例题已按书补全，每个例题配 markdown 说明（功能一句话、函数原型、参数注意），勘误与站点笔记 ch19.md 对齐。
> 运行环境：`cv2.imshow` 与鼠标回调、滚动条交互需**本地 Python 环境运行**（弹出 GUI 窗口），Jupyter 等无显示环境无法演示；运行前请确认工作目录为素材根目录（picture*.jpg 等素材都在这里）。
> 唯一的素材替换：例 19.13 原书读入 `lena512.bmp`，此处用 `picture1.jpg` 灰度读入顶替。


## 19.1 绘画基础

OpenCV 提供六个绘图函数：`cv2.line()` 绘制直线、`cv2.rectangle()` 绘制矩形、`cv2.circle()` 绘制圆、`cv2.ellipse()` 绘制椭圆、`cv2.polylines()` 绘制多边形、`cv2.putText()` 添加文字（对应例 19.1～19.8）。它们都往传入的"画布"数组上直接写像素，共有参数如下：

| 参数 | 含义 |
| --- | --- |
| `img` | 画布（canvas），绘制结果直接写进这个数组 |
| `color` | 颜色，按 **BGR** 顺序（`(0,255,0)` 是绿色）；灰度图像传灰度值 |
| `thickness` | 线条粗细，默认 1；**取 -1 表示填充（实心）** |
| `lineType` | 线条类型，默认 8 连接；可选 `cv2.LINE_4`（4 连接）/ `cv2.LINE_8`（8 连接）/ `cv2.LINE_AA`（抗锯齿，线条更平滑） |
| `shift` | 坐标数值的小数精度，一般不需要设置 |

> **勘误**：原书表 19-1 把 `cv2.FILLED` 列为 lineType 的取值。实测 `cv2.FILLED == -1`，它属于 **`thickness`** 参数的取值（-1 即填充）；`lineType` 实际只在 `LINE_4`、`LINE_8`、`LINE_AA` 三者中选。


### 例 19.1 · 黑色画布上画三条线（19.1.1 绘制直线）

用 `cv2.line()` 在 301×301 黑色画布上绘制三条颜色、粗细各异的线段：蓝色对角线（粗 3）、绿色水平细线（粗 1）、红色垂直粗线（粗 6）。

- 原型：`img = cv2.line(img, pt1, pt2, color[, thickness[, lineType]])`；`pt1`/`pt2` 为线段起点/终点
- 颜色按 BGR 顺序：`(255,0,0)` 是蓝色而非红色
- 绘图是就地修改：返回值就是原画布数组，`img = cv2.line(...)` 的赋值可写可不写；想在原图副本上试画需先 `img.copy()`


In [ ]:
# 例 19.1 黑色背景绘制三条线段
import numpy as np
import cv2

n = 300
img = np.zeros((n+1, n+1, 3), np.uint8)
img = cv2.line(img, (0, 0), (n, n), (255, 0, 0), 3)      # 蓝色对角线
img = cv2.line(img, (0, 100), (n, 100), (0, 255, 0), 1)  # 绿色水平线
img = cv2.line(img, (100, 0), (100, n), (0, 0, 255), 6)  # 红色垂直线

winname = 'Demo19.1'
cv2.namedWindow(winname)
cv2.imshow(winname, img)
cv2.waitKey(0)
cv2.destroyAllWindows()

### 例 19.2 · 白底画红色实心矩形（19.1.2 绘制矩形）

用 `cv2.rectangle()` 在白色画布上绘制红色实心矩形；实心与空心只差一个 `thickness`（换成 2 即两像素宽边框的空心矩形）。

- 原型：`img = cv2.rectangle(img, pt1, pt2, color[, thickness[, lineType]])`；`pt1`、`pt2` 是矩形的两个**对角顶点**
- `thickness=-1`（即 `cv2.FILLED`）表示填充成实心
- 原书窗口名写作 `Demo19.1`，按章节应为 `Demo19.2`（代码保持原书面貌）


In [ ]:
# 例 19.2 白色背景绘制实心矩形
import numpy as np
import cv2

n = 300
img = np.ones((n, n, 3), np.uint8) * 255
img = cv2.rectangle(img, (50, 50), (n-100, n-50), (0, 0, 255), -1)  # thickness=-1 实心
winname = 'Demo19.1'
cv2.namedWindow(winname)
cv2.imshow(winname, img)
cv2.waitKey(0)
cv2.destroyAllWindows()

### 例 19.3 · 绘制一组同心圆（19.1.3 绘制圆形）

用 `cv2.circle()` 以画布中心为圆心，画一组半径从 5 开始、公差 12 递增的红色同心圆（粗细固定 3）。

- 原型：`img = cv2.circle(img, center, radius, color[, thickness[, lineType]])`
- 圆心用 `round()` 保证是整数；写 `center=(d/2, d/2)` 这样的浮点坐标会报错
- `red = (0, 0, 255)`：BGR 顺序的红色（原书注释误写为"设置白色变量"）


In [ ]:
# 例 19.3 绘制一组同心圆
import numpy as np
import cv2

d = 400
img = np.ones((d, d, 3), dtype="uint8") * 255
(centerX, centerY) = (round(img.shape[1] / 2), round(img.shape[0] / 2))
# 将图像的中心作为圆心, 实际值为 d/2
red = (0, 0, 255)   # 原书注释写作"设置白色变量"，实为红色
for r in range(5, round(d/2), 12):
    cv2.circle(img, (centerX, centerY), r, red, 3)

cv2.imshow("Demo19.3", img)
cv2.waitKey(0)
cv2.destroyAllWindows()

### 例 19.4 · 绘制 100 个随机实心圆（19.1.3 绘制圆形）

用 `cv2.circle()` 配合 `np.random.randint` 绘制 100 个位置、大小、颜色均随机的实心圆。

- `np.random.randint` 的上界 `high` 取不到：`high=256` 才能产生 255 的颜色分量
- 随机颜色用 `.tolist()` 转成 Python 列表后再传给 `circle`
- `thickness=-1`：实心填充


In [ ]:
# 例 19.4 绘制一组位置和大小均随机的实心圆
import numpy as np
import cv2

d = 400
img = np.ones((d, d, 3), dtype="uint8") * 255      # 生成白色背景
for i in range(0, 100):
    centerX = np.random.randint(0, high=d)
    # 生成随机圆心 centerX, 确保在画布 img 内
    centerY = np.random.randint(0, high=d)
    # 生成随机圆心 centerY, 确保在画布 img 内
    radius = np.random.randint(5, high=d/5)
    # 生成随机半径, 值范围为 [5, d/5), 最大半径是 d/5
    color = np.random.randint(0, high=256, size=(3,)).tolist()
    # 生成随机颜色, 3 个 [0,256) 的随机数
    cv2.circle(img, (centerX, centerY), radius, color, -1)
    # 使用上述随机数在画布 img 内画圆（实心）

cv2.imshow("demo19.4", img)
cv2.waitKey(0)
cv2.destroyAllWindows()

### 例 19.5 · 绘制 10 个随机空心椭圆（19.1.4 绘制椭圆）

用 `cv2.ellipse()` 绘制中心、形状固定（横轴半径 100、竖轴半径 200），角度、颜色、粗细随机的 10 个空心椭圆。

- 原型：`img = cv2.ellipse(img, center, axes, angle, startAngle, endAngle, color[, thickness[, lineType]])`
- `axes` 为轴长（横轴半径、竖轴半径）；`center` 不可以是浮点（如 `center=(d/2, d/2)` 会报错）
- `startAngle=0, endAngle=360` 画完整椭圆；角度单位是度，沿 y 轴向下的坐标系顺时针增长，改小角度范围只画弧段


In [ ]:
# 例 19.5 随机绘制一组空心椭圆
import numpy as np
import cv2

d = 400
img = np.ones((d, d, 3), dtype="uint8") * 255      # 生成白色背景
center = (round(d/2), round(d/2))
# 注意数值类型, 不可以使用语句 center=(d/2, d/2)
size = (100, 200)   # 轴的长度
for i in range(0, 10):
    angle = np.random.randint(0, 361)     # 偏移角度
    color = np.random.randint(0, high=256, size=(3,)).tolist()
    # 生成随机颜色, 3 个 [0,256) 的随机数
    thickness = np.random.randint(1, 9)
    cv2.ellipse(img, center, size, angle, 0, 360, color, thickness)

cv2.imshow("demo19.5", img)
cv2.waitKey(0)
cv2.destroyAllWindows()

### 例 19.6 · 绘制封闭的绿色菱形（19.1.5 绘制多边形）

用 `cv2.polylines()` 绘制封闭多边形（菱形）；代码末尾注释给出 `isClosed=False` 的不封闭版本（仅把顶点依次连线、首尾不封口）。

- 原型：`img = cv2.polylines(img, pts, isClosed, color[, thickness[, lineType[, shift]]])`
- 顶点数组必须是 `numpy.int32` 类型、形状"顶点数 × 1 × 2"；`reshape((-1, 1, 2))` 的 -1 表示该维度由其他参数自动推算
- 第 2 个参数是"顶点数组的列表"，一次可画多条多边形
- `polylines` 只描边；要用同一组顶点画实心多边形，改用 `cv2.fillPoly(img, [pts], color)`


In [ ]:
# 例 19.6 绘制封闭的多边形（isClosed=False 则不封闭）
import numpy as np
import cv2

d = 400
img = np.ones((d, d, 3), dtype="uint8") * 255      # 生成白色背景
pts = np.array([[200, 50], [300, 200], [200, 350], [100, 200]], np.int32)
# 生成各个顶点, 注意数据类型为 int32
pts = pts.reshape((-1, 1, 2))
# 第 1 个参数为 -1, 表明它未设置具体值, 它所表示的维度值是通过其他参数值计算得到的
cv2.polylines(img, [pts], True, (0, 255, 0), 8)
# 调用函数 cv2.polylines() 完成多边形绘图。注意, 第 3 个参数控制多边形是否封闭

cv2.imshow("demo19.6", img)
cv2.waitKey(0)
cv2.destroyAllWindows()

# 不封闭版本（对应原书图 19-7）：
# cv2.polylines(img, [pts], False, (0, 255, 0), 8)

### 例 19.7 · 两遍叠画实现文字描边（19.1.6 在图形上绘制文字）

用 `cv2.putText()` 在同一位置先画粗细 15 的绿字、再叠画粗细 5 的红字，细笔画覆盖在粗笔画中间，形成"描边"（outline）效果。

- 原型：`img = cv2.putText(img, text, org, fontFace, fontScale, color[, thickness[, lineType[, bottomLeftOrigin]]])`
- `org` 是文字**左下角**起点：文字主体画在 org 上方，字母 p/g/y 的下伸部（descender）会越过 org 向下伸，排版时注意预留空间
- `fontFace` 为八种 Hershey 字体（仅支持 ASCII，写不了中文）；`fontScale` 是字号缩放倍数


In [ ]:
# 例 19.7 两遍叠画实现文字描边
import numpy as np
import cv2

d = 400
img = np.ones((d, d, 3), dtype="uint8") * 255      # 生成白色背景
font = cv2.FONT_HERSHEY_SIMPLEX
cv2.putText(img, 'OpenCV', (0, 200), font, 3, (0, 255, 0), 15)
cv2.putText(img, 'OpenCV', (0, 200), font, 3, (0, 0, 255), 5)

cv2.imshow("demo19.7", img)
cv2.waitKey(0)
cv2.destroyAllWindows()

### 例 19.8 · 绘制垂直镜像文字（19.1.6 在图形上绘制文字）

第 1 次调用画上方正常文字；第 2 次调用末位参数 `bottomLeftOrigin=True`，画出下方垂直镜像的文字。

- `putText` 参数顺序为 `..., color, thickness, lineType, bottomLeftOrigin`：原书把 `FONT_HERSHEY_SCRIPT_SIMPLEX`（值 6）放在了 lineType 槽上，字体并未真正切换成手写体（实测与 `LINE_8` 输出逐像素相同）；镜像效果不受影响——`True` 落在正确的 `bottomLeftOrigin` 槽上
- 想真正用手写体，应把 `fontFace` 形参换成 `cv2.FONT_HERSHEY_SCRIPT_SIMPLEX`
- 原书窗口名写作 `demo19.7`，按章节应为 `demo19.8`（代码保持原书面貌）


In [ ]:
# 例 19.8 绘制镜像文字（bottomLeftOrigin=True）
import numpy as np
import cv2

d = 400
img = np.ones((d, d, 3), dtype="uint8") * 255      # 生成白色背景
font = cv2.FONT_HERSHEY_SIMPLEX
cv2.putText(img, 'OpenCV', (0, 150), font, 3, (0, 0, 255), 15)
cv2.putText(img, 'OpenCV', (0, 250), font, 3, (0, 255, 0), 15,
            cv2.FONT_HERSHEY_SCRIPT_SIMPLEX, True)
# 注意：原书此行的 FONT_HERSHEY_SCRIPT_SIMPLEX 落在 lineType 槽上, 字体并未切换,
# 镜像效果来自最后的 True（bottomLeftOrigin）

cv2.imshow("demo19.7", img)
cv2.waitKey(0)
cv2.destroyAllWindows()

## 19.2 鼠标交互

把窗口里的单击、双击、拖曳等鼠标事件交给固定签名的回调函数处理（对应例 19.9～19.11）。

- 回调按固定格式创建：`def OnMouseAction(event, x, y, flags, param)`
  - `event`：触发的事件，如 `EVENT_LBUTTONDOWN` 按下左键、`EVENT_LBUTTONDBLCLK` 双击左键、`EVENT_MOUSEMOVE` 鼠标滑动等 12 种
  - `flags`：拖曳事件与键盘鼠标联合事件，如 `EVENT_FLAG_LBUTTON` 左键拖曳、`EVENT_FLAG_CTRLKEY` 按下 Ctrl 等 6 种；本质是位掩码（左键 1、右键 2、中键 4、Ctrl 8、Shift 16、Alt 32），组合状态用按位与检测
  - `param`：函数 ID，可通过 `setMouseCallback` 的第 3 个可选参数传入，给回调带参数
- `cv2.setMouseCallback(winname, onMouse)`：把响应函数与指定**窗口名**绑定，窗口须先 `namedWindow` 创建

> 注意：本节例题需**本地 GUI 环境**（弹出的 OpenCV 窗口）才能交互，Jupyter 等无显示环境无法演示。


### 例 19.9 · 判断触发了哪种鼠标事件（19.2 鼠标交互）

单击左键/右键/中键或按住左键拖动时，回调在控制台打印对应提示；代码末尾用 `[i for i in dir(cv2) if 'EVENT' in i]` 列出全部鼠标事件常量（共 18 个 = 12 种事件 + 6 个标志）。

- 回调签名固定五参数：`event` 判定单击/双击/移动，`flags` 判定拖曳与 Ctrl/Shift/Alt 组合
- `flags` 是位掩码，多状态可同时出现，严谨判断用按位与：`flags & cv2.EVENT_FLAG_CTRLKEY`
- `cv2.setMouseCallback(winname, onMouse)` 按**窗口名**绑定回调，窗口须先 `namedWindow` 创建
- 需本地 GUI 运行：鼠标回调依赖弹出的 OpenCV 窗口交互，Jupyter 等无显示环境无法演示


In [ ]:
# 例 19.9 对触发的鼠标事件进行判断
import cv2
import numpy as np

def Demo(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        print("单击了鼠标左键")
    elif event == cv2.EVENT_RBUTTONDOWN:
        print("单击了鼠标右键")
    elif flags == cv2.EVENT_FLAG_LBUTTON:
        print("按住左键拖动了鼠标")
    elif event == cv2.EVENT_MBUTTONDOWN:
        print("单击了中间键")

img = np.ones((300, 300, 3), np.uint8) * 255
cv2.namedWindow('Demo19.9')
cv2.setMouseCallback('Demo19.9', Demo)      # 将响应函数 Demo 与窗口建立连接（绑定）
cv2.imshow('Demo19.9', img)
cv2.waitKey()
cv2.destroyAllWindows()

# 查看OpenCV所支持的鼠标事件：
events = [i for i in dir(cv2) if 'EVENT' in i]
print(events)

### 例 19.10 · 双击画随机矩形（19.2.1 简单示例）

双击左键时以当前位置为顶点，绘制大小随机、颜色随机的空心矩形（粗细 2）；可在不同位置多次双击。

- 只响应 `cv2.EVENT_LBUTTONDBLCLK`（双击左键）；画布 `img` 是全局变量，回调里只读不写
- 交互主循环 `while(1) + imshow + waitKey(20)` 持续刷新窗口，Esc（ASCII 27）退出——没有心跳循环，回调画的内容不会出现在窗口里
- 需本地 GUI 运行


In [ ]:
# 例 19.10 双击鼠标绘制大小随机、颜色随机的矩形
import cv2
import numpy as np

d = 400

def draw(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDBLCLK:
        p1x = x
        p1y = y
        p2x = np.random.randint(1, d-50)
        p2y = np.random.randint(1, d-50)
        color = np.random.randint(0, high=256, size=(3,)).tolist()
        cv2.rectangle(img, (p1x, p1y), (p2x, p2y), color, 2)

img = np.ones((d, d, 3), dtype="uint8") * 255
cv2.namedWindow('Demo19.10')
cv2.setMouseCallback('Demo19.10', draw)
while(1):
    cv2.imshow('Demo19.10', img)
    if cv2.waitKey(20) == 27:      # Esc 退出
        break
cv2.destroyAllWindows()

### 例 19.11 · 键盘选模式 + 鼠标单击绘制（19.2.2 进阶示例）

键盘 `r/c/l/e/t` 切换矩形/圆/线段/椭圆/文字模式，`f/u` 切换实心/空心，单击左键按当前模式绘制，Esc 退出。

- `k = cv2.waitKey(1) & 0xFF`：按位与取返回值低 8 位（某些环境高位非零，直接与 `ord()` 比较会失配）；延时用 1ms 保证循环立即刷新，`waitKey(0)` 会无限期等待、循环卡死
- 全局变量 `mode`、`thickness` 在主循环里赋值、回调里只读，不写 `global` 也能正常工作
- 需本地 GUI 运行


In [ ]:
# 例 19.11 键盘与鼠标组合：r 矩形 c 圆 l 线段 e 椭圆 t 文字 f 实心 u 空心 Esc 退出
import cv2
import numpy as np

thickness = -1
mode = 1
d = 400

def draw_circle(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        a = np.random.randint(1, d-50)
        r = np.random.randint(1, d/5)
        angle = np.random.randint(0, 361)
        color = np.random.randint(0, high=256, size=(3,)).tolist()
        if mode == 1:
            cv2.rectangle(img, (x, y), (a, a), color, thickness)
        elif mode == 2:
            cv2.circle(img, (x, y), r, color, thickness)
        elif mode == 3:
            cv2.line(img, (a, a), (x, y), color, 3)
        elif mode == 4:
            cv2.ellipse(img, (x, y), (100, 150), angle, 0, 360, color, thickness)
        elif mode == 5:
            cv2.putText(img, 'OpenCV', (0, round(d/2)),
                        cv2.FONT_HERSHEY_SIMPLEX, 2, color, 5)

img = np.ones((d, d, 3), np.uint8) * 255
cv2.namedWindow('image')
cv2.setMouseCallback('image', draw_circle)
while(1):
    cv2.imshow('image', img)
    k = cv2.waitKey(1) & 0xFF
    if k == ord('r'):
        mode = 1
    elif k == ord('c'):
        mode = 2
    elif k == ord('l'):
        mode = 3
    elif k == ord('e'):
        mode = 4
    elif k == ord('t'):
        mode = 5
    elif k == ord('f'):
        thickness = -1
    elif k == ord('u'):
        thickness = 3
    elif k == 27:
        break
cv2.destroyAllWindows()

## 19.3 滚动条

滚动条（Trackbar）依附于特定窗口而存在，通过调节滚动条能够设置、获取指定范围内的特定值（对应例 19.12～19.14）。

- `cv2.createTrackbar(trackbarname, winname, value, count, onChange)`：定义滚动条；`value` 初始值（决定滑块位置）、`count` 最大值（最小值通常为 0）、`onChange` 为滚动条变化时触发的回调
- `cv2.getTrackbarPos(trackbarname, winname)`：随时获取滚动条当前值
- 三种典型用法：三滚动条调 RGB 模拟调色板（例 19.12）、实时调阈值处理参数（例 19.13）、0/1 两档当开关（例 19.14）

> 注意：本节例题需**本地 GUI 环境**才能交互，Jupyter 等无显示环境无法演示。


### 例 19.12 · 三滚动条模拟调色板（19.3.1 用滚动条实现调色板）

R、G、B 三根 0～255 的滚动条分别设定三个颜色分量，拖动任一根，画布立即刷成对应的混色。

- 回调 `changeColor(x)` 只在滚动条变化时触发一次，`x` 只是触发变化那一根的值；其余分量用 `cv2.getTrackbarPos(name, winname)` 现读——三根全读最省心
- 整块刷色 `img[:] = [b, g, r]`：注意 OpenCV 是 **BGR** 顺序
- 需本地 GUI 运行


In [ ]:
# 例 19.12 滚动条模拟调色板
import cv2
import numpy as np

def changeColor(x):
    r = cv2.getTrackbarPos('R', 'image')
    g = cv2.getTrackbarPos('G', 'image')
    b = cv2.getTrackbarPos('B', 'image')
    img[:] = [b, g, r]      # 注意 BGR 顺序

img = np.zeros((100, 700, 3), np.uint8)
cv2.namedWindow('image')
cv2.createTrackbar('R', 'image', 0, 255, changeColor)
cv2.createTrackbar('G', 'image', 0, 255, changeColor)
cv2.createTrackbar('B', 'image', 0, 255, changeColor)
while(1):
    cv2.imshow('image', img)
    k = cv2.waitKey(1) & 0xFF
    if k == 27:
        break
cv2.destroyAllWindows()

### 例 19.13 · 滚动条控制阈值处理的类型与阈值（19.3.2 用滚动条控制阈值处理参数）

Type 滚动条（0～4 依次对应 `THRESH_BINARY`、`THRESH_BINARY_INV`、`THRESH_TRUNC`、`THRESH_TOZERO`、`THRESH_TOZERO_INV`）与 Value 滚动条（0～255）实时控制 `cv2.threshold()` 的处理结果。

- 两个滚动条各挂独立回调 `onType`/`onValue`，逻辑相同：读双条当前值 → `cv2.threshold(o, Value, 255, Type)` → 显示结果
- 回调内对 `Type`、`Value` 直接赋值却未声明 `global`，"碰巧能跑"：赋值后立即在局部使用；一旦在赋值前先读就会触发 `UnboundLocalError`，规范写法是函数内声明 `global Type, Value`
- 原书读入 `lena512.bmp`，此处用 `picture1.jpg` 灰度读入顶替；运行时工作目录需为素材根目录
- 需本地 GUI 运行


In [ ]:
# 例 19.13 滚动条控制阈值处理的阈值与模式
# 原书读入 lena512.bmp，此处用 picture1.jpg 灰度读入顶替
import cv2

Type = 0    # 阈值处理方式
Value = 0   # 使用的阈值

def onType(a):
    Type = cv2.getTrackbarPos(tType, windowName)
    Value = cv2.getTrackbarPos(tValue, windowName)
    ret, dst = cv2.threshold(o, Value, 255, Type)
    cv2.imshow(windowName, dst)

def onValue(a):
    Type = cv2.getTrackbarPos(tType, windowName)
    Value = cv2.getTrackbarPos(tValue, windowName)
    ret, dst = cv2.threshold(o, Value, 255, Type)
    cv2.imshow(windowName, dst)

o = cv2.imread("picture1.jpg", 0)
windowName = "Demo19.13"           # 窗体名
cv2.namedWindow(windowName)
cv2.imshow(windowName, o)

tType = "Type"      # 用来选取阈值处理方式的滚动条
tValue = "Value"    # 用来选取阈值的滚动条
cv2.createTrackbar(tType, windowName, 0, 4, onType)
cv2.createTrackbar(tValue, windowName, 0, 255, onValue)
if cv2.waitKey(0) == 27:
    cv2.destroyAllWindows()

### 例 19.14 · 滚动条当"实心/空心"开关（19.3.3 用滚动条作为开关）

滚动条只有 0/1 两档：0 → `thickness=-1` 画实心矩形，1 → `thickness=2` 画空心矩形；调整开关后再双击左键即可切换效果。

- "值域 0～1 + 空回调"是轻量开关套路：回调 `fill(x)` 留空，主循环里 `cv2.getTrackbarPos` 读取后切换状态
- 开关的 0/1 也可代表任意两种状态（如显示原图/显示处理结果）；滚动条名 `R` 只是原书沿用的名字，自定义程序建议取语义化名字（如 `Fill`）
- 需本地 GUI 运行


In [ ]:
# 例 19.14 滚动条作为开关：控制矩形实心/空心
import cv2
import numpy as np

d = 400
global thickness      # 原书如此（模块级声明，实际多余）
thickness = -1

def fill(x):
    pass

def draw(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDBLCLK:
        p1x = x
        p1y = y
        p2x = np.random.randint(1, d-50)
        p2y = np.random.randint(1, d-50)
        color = np.random.randint(0, high=256, size=(3,)).tolist()
        cv2.rectangle(img, (p1x, p1y), (p2x, p2y), color, thickness)

img = np.ones((d, d, 3), np.uint8) * 255
cv2.namedWindow('image')
cv2.setMouseCallback('image', draw)
cv2.createTrackbar('R', 'image', 0, 1, fill)   # 原书将开关滚动条命名为 R
while(1):
    cv2.imshow('image', img)
    k = cv2.waitKey(1) & 0xFF
    g = cv2.getTrackbarPos('R', 'image')
    if g == 0:
        thickness = -1
    else:
        thickness = 2
    if k == 27:
        break
cv2.destroyAllWindows()